In [1]:
import pandas as pd
import geopandas as gpd
import folium
from geopy.distance import geodesic
from rapidfuzz import process, fuzz
import matplotlib.pyplot as plt
import numpy as np
from pygris import tracts

In [3]:
stops = pd.read_csv('filtered_crimes2.csv')

In [19]:
stops

,Incident Datetime,Incident Date,Incident Time,Incident Year,Incident Day of Week,Report Datetime,Row ID,Incident ID,Incident Number,CAD Number,...,Longitude,Point,Neighborhoods,ESNCAG - Boundary File,Central Market/Tenderloin Boundary Polygon - Updated,Civic Center Harm Reduction Project Boundary,HSOC Zones as of 2018-06-05,Invest In Neighborhoods (IIN) Areas,Current Supervisor Districts,Current Police Districts
0,2024/09/05 06:15:00 AM,2024/09/05,06:15 AM,2024,Thursday,2024/09/05 07:10:00 AM,142126304134,1421263,240557895,242490594.0,...,-122.409309,POINT (-122.40930938720703 37.78434753417969),20.0,NaN,1.0,1.0,NaN,NaN,10.0,5.0
1,2024/05/08 03:09:00 PM,2024/05/08,03:09 PM,2024,Wednesday,2024/05/08 09:28:00 PM,138885406301,1388854,240290863,241292953.0,...,-122.411720,POINT (-122.4117202758789 37.79629135131836),107.0,NaN,NaN,NaN,NaN,NaN,3.0,6.0
2,2024/05/07 04:15:00 PM,2024/05/07,04:15 PM,2024,Tuesday,2024/05/07 08:39:00 PM,138857806314,1388578,240288476,241282340.0,...,-122.407692,POINT (-122.4076919555664 37.7801628112793),32.0,NaN,1.0,1.0,1.0,NaN,10.0,1.0
3,2024/05/07 05:05:00 PM,2024/05/07,05:05 PM,2024,Tuesday,2024/05/08 04:10:00 PM,138876806373,1388768,240290108,241292157.0,...,-122.372658,POINT (-122.3726577758789 37.824119567871094),36.0,NaN,NaN,NaN,NaN,NaN,10.0,1.0
4,2024/05/08 05:25:00 PM,2024/05/08,05:25 PM,2024,Wednesday,2024/05/08 05:25:00 PM,138879512080,1388795,240290465,241292434.0,...,-122.498360,POINT (-122.49835968017578 37.71369552612305),43.0,NaN,NaN,NaN,NaN,NaN,8.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6819,2024/12/30 05:00:00 PM,2024/12/30,05:00 PM,2024,Monday,2024/12/31 02:30:00 PM,145747906244,1457479,246169204,NaN,...,-122.408401,POINT (-122.40840148925781 37.788291931152344),19.0,NaN,NaN,NaN,NaN,NaN,3.0,6.0
6820,2024/12/30 05:23:00 PM,2024/12/30,05:23 PM,2024,Monday,2024/12/30 09:18:00 PM,145749706374,1457497,246169248,NaN,...,-122.405663,POINT (-122.4056625366211 37.80667495727539),99.0,NaN,NaN,NaN,NaN,NaN,3.0,6.0
6821,2024/12/26 05:45:00 PM,2024/12/26,05:45 PM,2024,Thursday,2024/12/30 02:49:00 PM,145749506224,1457495,246169191,NaN,...,-122.396584,POINT (-122.3965835571289 37.79458999633789),108.0,NaN,NaN,NaN,NaN,NaN,3.0,6.0
6822,2024/12/18 05:04:00 PM,2024/12/18,05:04 PM,2024,Wednesday,2024/12/18 09:11:00 PM,146280506372,1462805,246169408,NaN,...,-122.405609,POINT (-122.40560913085938 37.716590881347656),75.0,NaN,NaN,NaN,NaN,NaN,9.0,9.0


In [4]:
stop2s = pd.read_csv('results2.csv')

In [15]:
stop2s

,STOPID,STOPNAME,LATITUDE,School Name,LONGITUDE,time_difference,route_id,NAMELSAD
0,5351,Mansell St&Somerset St S-NS/PS,37.720418,"Burton, Phillip And Sala Burton High School",-122.405094,0 min 27 sec,29,Census Tract 259
1,4783,Gilman Ave&3rd St E-NS/PS,37.722458,"Burton, Phillip And Sala Burton High School",-122.395415,4 min 6 sec,29,Census Tract 234
2,3071,Balboa St&Park Presidio Blvd SW-NS,37.776788,Gateway High School / Kipp Sf Bay Academy,-122.472411,11 min 51 sec,31,Census Tract 476
3,5274,Lombard St&Fillmore St SE-FS/BZ,37.799818,Galileo High School,-122.435878,7 min 3 sec,28,Census Tract 129.02
4,4329,Castro St&25TH St SE-NS/BZ,37.749600,Gateway High School / Kipp Sf Bay Academy,-122.433830,15 min 46 sec,24,Census Tract 214
...,...,...,...,...,...,...,...,...
788,6609,Sutter St&Steiner St NE-NS/BZ,37.785942,Gateway High School / Kipp Sf Bay Academy,-122.434793,1 min 57 sec,2,Census Tract 152.02
789,5633,Galvez Ave&Hill Ave MB-NS/BZ,37.728795,Galileo High School,-122.367027,53 min 8 sec,19,Census Tract 9806
790,7979,Moscow St&Geneva Ave S-FS/BZ,37.713233,"Burton, Phillip And Sala Burton High School",-122.433557,11 min 38 sec,54,Census Tract 263.02
791,3603,46th Ave&Vicente St SE-NS/PS,37.737973,"Washington, George Washington High School",-122.504268,19 min 18 sec,18,Census Tract 354


In [5]:
# Function to calculate distance between two coordinates
def is_within_distance(coord1, coord2, max_distance_ft):
    distance_m = geodesic(coord1, coord2).feet
    return distance_m <= max_distance_ft

In [6]:

from geopy.distance import geodesic

In [20]:
results = []
for index1, row1 in stops.iterrows():
    coord1 = (row1['LATITUDE'], row1['LONGITUDE'])
    for index2, row2 in df2.iterrows():
        # Check if Latitude and Longitude are valid before calculating distance
        if pd.notna(row2['Latitude']) and pd.notna(row2['Longitude']):
            coord2 = (row2['Latitude'], row2['Longitude'])
            if is_within_distance(coord1, coord2, 25):  # Check if within 15ft
                results.append({
                    'df_index': index1,
                    'df2_index': index2,
                    'df_coordinates': coord1,
                    'df2_coordinates': coord2
                })
        #else:

KeyError: 'STOPID'

In [ ]:
# Convert results to a DataFrame
results_df = pd.DataFrame(results)

In [ ]:
results_df

In [ ]:
# Save the results
results_df.to_csv("crimey.csv", index=False)